# 04 - NMO velocity sensitivity

The NMO velocity field is perturbed by a uniform multiplicative factor of 0, +/-5 and
+/-10 percent before stacking, and the already-trained network is applied without
retraining. Quantifies how velocity-model error propagates into the reconstruction.

## 1. Generate perturbed stacks

In [ ]:
# NMO velocity-error sensitivity (Part A: generate perturbed stacks) ---
# Reviewer #1: "run a sensitivity test where NMO uses a perturbed velocity model
# (e.g., ±5%, ±10%) to quantify how velocity uncertainty propagates into the
# reconstruction."
#
# NOTE: process_and_stack_dataset() reads the NMO velocity from the GLOBAL variable
# `baseline_data`. This cell temporarily overwrites it and ALWAYS restores it.

import os
os.makedirs('outputs/velocity_sensitivity/data_cache', exist_ok=True)

PERTURBATIONS = [0.0, +0.05, -0.05, +0.10, -0.10]
_original_baseline_data = baseline_data.copy()
velocity_stacks = {}

try:
    for pct in PERTURBATIONS:
        label = "0%" if pct == 0 else f"{pct*100:+.0f}%"
        print(f"\n=== NMO velocity perturbation: {label} ===")

        # Perturb the velocity model used for NMO only -- the recorded wavefield
        # (scatter_data) is unchanged, exactly as in the field where the data is what
        # it is and only our velocity estimate is wrong.
        baseline_data = _original_baseline_data * (1.0 + pct)

        stack_pert = process_and_stack_dataset(
            seismic_data=scatter_data,
            dataset_name=f"VelPerturb_{label}",
            source_locations_for_this_data=scatter_locs,
            save_dir_figures='outputs/velocity_sensitivity/figures',
            save_dir_numpy='outputs/velocity_sensitivity/stacked')
        velocity_stacks[label] = normalize_data(stack_pert.T)
finally:
    baseline_data = _original_baseline_data   # ALWAYS restore
    print("\nRestored the true baseline_data.")

# Save for the TensorFlow half
for label, arr in velocity_stacks.items():
    safe = label.replace('%','pct').replace('+','p').replace('-','m')
    np.save(f'outputs/velocity_sensitivity/data_cache/stack_{safe}.npy', arr)
np.save('outputs/velocity_sensitivity/data_cache/Y_full.npy', Y_full)

print(f"\nSaved {len(velocity_stacks)} perturbed stacks + Y_full.")
print(">>> Restart the kernel, then run Part B.")

## 2. Evaluate

In [ ]:
# evaluate the trained U-Net on each perturbed stack ---
import numpy as np, tensorflow as tf, matplotlib.pyplot as plt, json, os
from skimage.metrics import structural_similarity as ssim

WINDOW_SIZE, STEP_SIZE = 128, 5

def sliding_window_view(arr, window_shape, step_size):
    wh, ww = window_shape if isinstance(window_shape, tuple) else (window_shape, window_shape)
    sy, sx = step_size if isinstance(step_size, tuple) else (step_size, step_size)
    h, w = arr.shape
    shape = ((h - wh)//sy + 1, (w - ww)//sx + 1, wh, ww)
    st_y, st_x = arr.strides
    return np.lib.stride_tricks.as_strided(arr, shape=shape, strides=(st_y*sy, st_x*sx, st_y, st_x))

def reconstruct_from_patches_average(patches, output_shape, window_size, step_size):
    oh, ow = output_shape
    recon = np.zeros(output_shape, dtype=np.float32)
    count = np.zeros(output_shape, dtype=np.float32)
    patches = np.squeeze(patches)
    ny, nx_ = (oh - window_size)//step_size + 1, (ow - window_size)//step_size + 1
    idx = 0
    for yi in range(ny):
        for xi in range(nx_):
            if idx >= len(patches): break
            y0, x0 = yi*step_size, xi*step_size
            recon[y0:y0+window_size, x0:x0+window_size] += patches[idx]
            count[y0:y0+window_size, x0:x0+window_size] += 1.0
            idx += 1
        if idx >= len(patches): break
    count[count == 0] = 1.0
    return recon / count

def calculate_nrms(a, b):
    rms = np.sqrt(np.mean((a-b)**2))
    den = np.sqrt(np.mean(a**2)) + np.sqrt(np.mean(b**2))
    return 0.0 if den == 0 else 200.0*rms/den

MODEL_PATH = 'models/unet_32filters.keras'   # <-- your official 32-filter model
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print(f"Loaded {MODEL_PATH} ({model.count_params():,} params).")

Y_full = np.load('outputs/velocity_sensitivity/data_cache/Y_full.npy')
labels = ["0%", "+5%", "-5%", "+10%", "-10%"]

results, predictions = {}, {}
for label in labels:
    safe = label.replace('%','pct').replace('+','p').replace('-','m')
    X = np.load(f'outputs/velocity_sensitivity/data_cache/stack_{safe}.npy')

    view = sliding_window_view(X, (WINDOW_SIZE, WINDOW_SIZE), STEP_SIZE)
    patches = view.reshape(-1, WINDOW_SIZE, WINDOW_SIZE).copy()[..., np.newaxis]
    pred = reconstruct_from_patches_average(model.predict(patches, verbose=0),
                                            X.shape, WINDOW_SIZE, STEP_SIZE)
    predictions[label] = pred

    dr = Y_full.max() - Y_full.min()
    results[label] = {
        'input_nrms':  calculate_nrms(X, Y_full),
        'input_ssim':  ssim(Y_full, X, data_range=dr),
        'unet_nrms':   calculate_nrms(pred, Y_full),
        'unet_ssim':   ssim(Y_full, pred, data_range=dr),
    }

print(f"\n{'Velocity error':16s} {'Input NRMS':>11s} {'U-Net NRMS':>11s} {'Input SSIM':>11s} {'U-Net SSIM':>11s}")
for label in labels:
    r = results[label]
    print(f"{label:16s} {r['input_nrms']:10.2f}% {r['unet_nrms']:10.2f}% "
          f"{r['input_ssim']:11.4f} {r['unet_ssim']:11.4f}")

with open('outputs/velocity_sensitivity/sensitivity_results.json', 'w') as f:
    json.dump(results, f, indent=2)

# --- Plot: degradation vs. velocity error ---
pcts = [0, 5, -5, 10, -10]
order = sorted(range(len(labels)), key=lambda i: pcts[i])
xs = [pcts[i] for i in order]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(xs, [results[labels[i]]['unet_nrms'] for i in order], 'o-', label='U-Net output')
ax1.plot(xs, [results[labels[i]]['input_nrms'] for i in order], 's--', label='Unprocessed input')
ax1.set_xlabel('NMO velocity error (%)'); ax1.set_ylabel('NRMS (%)')
ax1.set_title('NRMS vs. velocity error'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(xs, [results[labels[i]]['unet_ssim'] for i in order], 'o-', label='U-Net output')
ax2.plot(xs, [results[labels[i]]['input_ssim'] for i in order], 's--', label='Unprocessed input')
ax2.set_xlabel('NMO velocity error (%)'); ax2.set_ylabel('SSIM')
ax2.set_title('SSIM vs. velocity error'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
os.makedirs('outputs/velocity_sensitivity/figures', exist_ok=True)
plt.savefig('outputs/velocity_sensitivity/figures/velocity_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Visual comparison of the U-Net output at each velocity error ---
fig, axes = plt.subplots(1, len(labels), figsize=(5*len(labels), 5), sharey=True)
vabs = np.percentile(np.abs(Y_full), 99)
for ax, i in zip(axes, order):
    label = labels[i]
    ax.imshow(predictions[label], cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs)
    ax.set_title(f"{label}\nNRMS={results[label]['unet_nrms']:.1f}%")
    ax.set_xlabel('CMP index')
axes[0].set_ylabel('Sample index')
plt.suptitle('U-Net output under NMO velocity error', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/velocity_sensitivity/figures/velocity_sensitivity_sections.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Figure

In [ ]:
import numpy as np, matplotlib.pyplot as plt, os

PERTURB_MAP = {"0%": 0.0, "+5%": 0.05, "-5%": -0.05, "+10%": 0.10, "-10%": -0.10}
plot_order = ["-10%", "-5%", "0%", "+5%", "+10%"]
ncol = len(plot_order) + 1

fig, axes = plt.subplots(2, ncol, figsize=(16, 6), sharex=True, sharey=True)
vabs = np.percentile(np.abs(Y_full), 99)

for j, label in enumerate(plot_order):
    safe = label.replace('%', 'pct').replace('+', 'p').replace('-', 'm')
    X = np.load(f'outputs/velocity_sensitivity/data_cache/stack_{safe}.npy')

    ax = axes[0, j]
    ax.imshow(X, cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs)
    ax.set_title(f"{label} velocity error\nNRMS = {results[label]['input_nrms']:.1f}%, "
                 f"SSIM = {results[label]['input_ssim']:.3f}", fontsize=10,
                 fontweight='bold' if label == '0%' else 'normal')

    ax = axes[1, j]
    im = ax.imshow(predictions[label], cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs)
    ax.set_title(f"NRMS = {results[label]['unet_nrms']:.1f}%, "
                 f"SSIM = {results[label]['unet_ssim']:.3f}", fontsize=10)
    ax.set_xlabel('CMP number', fontsize=10)

# Reference column
axes[0, -1].imshow(Y_full, cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs)
axes[0, -1].set_title('Reference\ndense target', fontsize=10, fontweight='bold')
axes[1, -1].axis('off')
for spine in axes[0, -1].spines.values():
    spine.set_edgecolor('black'); spine.set_linewidth(2.0)

axes[0, 0].set_ylabel('(a) Stacked input\nTime sample', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('(b) U-Net output\nTime sample', fontsize=11, fontweight='bold')

# Outline the true-velocity column
center = plot_order.index("0%")
for row in range(2):
    for spine in axes[row, center].spines.values():
        spine.set_edgecolor('black'); spine.set_linewidth(2.0)

cb = fig.colorbar(im, ax=axes, fraction=0.015, pad=0.02)
cb.set_label('Normalized amplitude', fontsize=11)

os.makedirs('outputs/velocity_sensitivity/figures', exist_ok=True)
fig.savefig('outputs/velocity_sensitivity/figures/velocity_sensitivity1.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()